# 18 RNN基础 - 循环神经网络与BPTT

本教程将带你理解**循环神经网络（RNN）**的基本原理，以及它如何处理**序列数据**。这是通向Transformer的必经之路。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、为什么需要RNN？

### 前馈网络的局限

之前学的全连接网络（Feed-Forward）有个重要假设：输入和输出之间**相互独立**。

但很多实际任务中，数据是**有序列关系的**：

| 任务类型 | 输入 | 输出 | 例子 |
|---------|------|------|------|
| 序列→值 | 序列 | 单个值 | 情感分析（句子→正面/负面） |
| 序列→序列 | 序列 | 序列 | 机器翻译（中文→英文） |
| 值→序列 | 单个值 | 序列 | 图像描述（图片→文字） |
| 序列预测 | 序列的一部分 | 序列的其余部分 | 文本补全 |

### RNN的核心思想

**利用隐藏状态（Hidden State）记录历史信息**

RNN在每个时间步：
1. 接收当前输入 $x_t$
2. 结合上一时刻的隐藏状态 $h_{t-1}$
3. 产生新的隐藏状态 $h_t$ 和输出 $y_t$

这使得RNN具有**记忆**能力。

In [ ]:
# 可视化对比前馈网络和RNN的处理方式
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 前馈网络
ax = axes[0]
ax.set_title('前馈网络\n(每个输入独立处理)', fontsize=12)
inputs = ['I', 'love', 'AI']
x_pos = [0, 1, 2]
for i, (x, word) in enumerate(zip(x_pos, inputs)):
    ax.text(x, 0.5, word, ha='center', va='center', fontsize=14, 
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    ax.arrow(x, 0.35, 0, -0.2, head_width=0.1, head_length=0.05, fc='red', ec='red')
    ax.text(x, -0.1, f'f(x{i})', ha='center', va='center', fontsize=12, color='red')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.3, 0.8)
ax.axis('off')
ax.text(1, -0.25, 'Problem: outputs are independent, no context', ha='center', 
        fontsize=10, color='red', style='italic')

# RNN
ax = axes[1]
ax.set_title('RNN\n(保留隐藏状态，有记忆)', fontsize=12)
for i, word in enumerate(inputs):
    x = i * 1.2
    # 输入
    ax.text(x, 0.5, word, ha='center', va='center', fontsize=14,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    # RNN单元
    ax.text(x, 0, 'RNN', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='circle', facecolor='lightyellow', alpha=0.7))
    # 输出
    ax.arrow(x, -0.15, 0, -0.15, head_width=0.08, head_length=0.04, fc='red', ec='red')
    ax.text(x, -0.5, f'h{i}', ha='center', va='center', fontsize=12, color='red')
    # 隐藏状态传递
    if i < len(inputs) - 1:
        ax.annotate('', xy=(x+1.2, 0.1), xytext=(x+0.15, 0),
                   arrowprops=dict(arrowstyle='->', color='green', lw=2))
        ax.text(x+0.6, 0.15, 'h', ha='center', fontsize=10, color='green')

ax.set_xlim(-0.5, 3.0)
ax.set_ylim(-0.7, 0.8)
ax.axis('off')
ax.text(1.5, -0.65, 'Advantage: each output contains info from all previous inputs', ha='center',
        fontsize=10, color='green', style='italic')

plt.tight_layout()
plt.show()

## 二、RNN的前向传播

### 核心公式

在每个时间步 $t$，RNN做以下计算：

**隐藏状态更新：**
$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

**输出（可选）：**
$$y_t = W_{hq} h_t + b_{hq}$$

### 参数说明

| 符号 | 含义 | 形状 |
|------|------|------|
| $x_t$ | 当前时间步输入 | (input_size,) |
| $h_t$ | 当前隐藏状态 | (hidden_size,) |
| $h_{t-1}$ | 上一时刻隐藏状态 | (hidden_size,) |
| $W_{ih}$ | 输入→隐藏权重 | (hidden_size, input_size) |
| $W_{hh}$ | 隐藏→隐藏权重 | (hidden_size, hidden_size) |
| $b_{ih}, b_{hh}$ | 偏置 | (hidden_size,) |

### 关键理解

RNN的**同一个单元**（同一组参数）在每个时间步**重复使用**，就像循环一样。
这也是"循环"神经网络名字的由来。

In [ ]:
# RNN前向传播 - 从零实现
print("=" * 70)
print("RNN前向传播 - 详细数值示例")
print("=" * 70)
print()

# 定义RNN参数
input_size = 3
hidden_size = 2
seq_length = 4

# 权重矩阵（给定值）
W_ih = np.array([[0.1, 0.2, 0.3],   # 输入→隐藏
                  [0.4, 0.5, 0.6]])  # shape: (hidden_size, input_size)
b_ih = np.array([0.1, -0.1])         # 偏置
W_hh = np.array([[0.7, 0.8],        # 隐藏→隐藏
                  [0.9, 1.0]])       # shape: (hidden_size, hidden_size)
b_hh = np.array([0.0, 0.1])          # 偏置

# 输入序列: 4个时间步，每个时间步3维
sequence = np.array([
    [1.0, 0.0, 0.0],  # t=0
    [0.0, 1.0, 0.0],  # t=1
    [0.0, 0.0, 1.0],  # t=2
    [1.0, 1.0, 0.0],  # t=3
])

print(f"输入序列 (seq_len={seq_length}, input_size={input_size}):")
for t, x in enumerate(sequence):
    print(f"  t={t}: {x}")
print()

# 前向传播
h = np.zeros(hidden_size)  # 初始隐藏状态 h_0 = 0
hidden_states = [h.copy()]

print(f"{'t':<4} {'输入 x_t':<20} {'W_ih·x_t':<18} {'W_hh·h':<18} {'z = 和':<18} {'h_t = tanh(z)':>15}")
print("-" * 100)

for t, x in enumerate(sequence):
    # RNN公式: h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)
    input_part = W_ih @ x + b_ih
    hidden_part = W_hh @ h + b_hh
    z = input_part + hidden_part
    h = np.tanh(z)
    hidden_states.append(h.copy())
    
    print(f"{t:<4} {str(x):<20} {str(np.round(input_part,4)):<18} {str(np.round(hidden_part,4)):<18} {str(np.round(z,4)):<18} {str(np.round(h,4)):>15}")

print()
print(f"最终隐藏状态: h_{seq_length} = {h}")

In [ ]:
# 用PyTorch的nn.RNN验证手动计算
print("\n=== 用PyTorch nn.RNN验证 ===")

torch.manual_seed(42)

# 创建PyTorch RNN
rnn = nn.RNN(input_size=3, hidden_size=2, batch_first=True)

# 手动设置权重（和上面一致）
with torch.no_grad():
    rnn.weight_ih_l0.copy_(torch.from_numpy(W_ih))
    rnn.bias_ih_l0.copy_(torch.from_numpy(b_ih))
    rnn.weight_hh_l0.copy_(torch.from_numpy(W_hh))
    rnn.bias_hh_l0.copy_(torch.from_numpy(b_hh))

# 输入 (batch=1, seq_len=4, input_size=3)
x_torch = torch.from_numpy(sequence).unsqueeze(0).float()

# 前向传播
output, h_n = rnn(x_torch)

print(f"PyTorch RNN输出:")
print(f"  output shape: {output.shape}")
print(f"  output (各时间步的隐藏状态):")
for t in range(seq_length):
    print(f"    t={t}: {output[0, t].detach().numpy()}")
print(f"  h_n (最终隐藏状态): {h_n[0].detach().numpy()}")

print(f"\n手动计算的最终隐藏状态: {h}")
print(f"PyTorch的最终隐藏状态:    {h_n[0].detach().numpy()}")
print(f"两者是否一致: {np.allclose(h, h_n[0].detach().numpy(), atol=1e-5)}")

## 三、通过时间的反向传播（BPTT）

### 什么是BPTT？

**BPTT (Backpropagation Through Time)** 是RNN特有的反向传播算法。

### 核心思想

将RNN**按时间步展开（Unroll）**，变成一个深层前馈网络，然后应用标准的反向传播。

```
展开前（循环）:              展开后（链式）:

    ┌─────┐                 x0     x1     x2     x3
    │     │                 │      │      │      │
 x─→│ RNN │──→ h            h0───→h1───→h2───→h3──→h4
    │     │                  ↓      ↓      ↓      ↓
    └─────┘                 y0     y1     y2     y3
      ↑
     h (循环)
```

### 梯度计算

假设损失函数为 $L = \sum_{t=0}^{T} L_t$，其中 $L_t$ 是时间步 $t$ 的损失。

隐藏状态的梯度：

$$\frac{\partial L}{\partial h_t} = \frac{\partial L_t}{\partial h_t} + \frac{\partial L}{\partial h_{t+1}} \cdot \frac{\partial h_{t+1}}{\partial h_t}$$

其中第二项说明 $h_t$ 的梯度来自：
1. 当前时间步的损失 $L_t$ 
2. 下一时刻 $h_{t+1}$ 传回来的梯度

### 权重 $W_{hh}$ 的梯度

由于 $W_{hh}$ 在所有时间步**共享**，它的梯度是各时间步梯度的**总和**：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}$$

In [ ]:
# BPTT数值演示
print("=" * 70)
print("BPTT (通过时间的反向传播) - 数值示例")
print("=" * 70)
print()

# 使用前面前向传播的结果
torch.manual_seed(42)

# 简化RNN: 使用PyTorch并逐步展开
input_size = 3
hidden_size = 2

rnn = nn.RNN(input_size, hidden_size, batch_first=True)

with torch.no_grad():
    rnn.weight_ih_l0.fill_(0.3)
    rnn.bias_ih_l0.fill_(0.1)
    rnn.weight_hh_l0.fill_(0.2)
    rnn.bias_hh_l0.fill_(0.05)

# 短序列
x = torch.tensor([[[1.0, 0.0, 0.0],
                   [0.0, 1.0, 0.0],
                   [0.0, 0.0, 1.0]]], dtype=torch.float32)
target = torch.tensor([[[0.5, -0.5],
                        [0.3, -0.3],
                        [0.1, -0.1]]], dtype=torch.float32)

# 前向传播
output, h_n = rnn(x)

# 计算MSE损失
loss = nn.MSELoss()(output, target)
print(f"前向传播:")
for t in range(3):
    print(f"  t={t}: h_{t+1} = {output[0,t].detach().numpy()}, target = {target[0,t].numpy()}")
print(f"\n损失 L = {loss.item():.6f}")
print()

# 反向传播
loss.backward()

print(f"反向传播 - 参数梯度:")
print(f"  W_ih (输入→隐藏权重) 梯度范数: {rnn.weight_ih_l0.grad.norm().item():.6f}")
print(f"  W_hh (隐藏→隐藏权重) 梯度范数: {rnn.weight_hh_l0.grad.norm().item():.6f}")
print(f"  b_ih 梯度范数: {rnn.bias_ih_l0.grad.norm().item():.6f}")
print(f"  b_hh 梯度范数: {rnn.bias_hh_l0.grad.norm().item():.6f}")
print()

# 逐步展开看各时间步的梯度贡献
print("各时间步对 W_hh 梯度的贡献:")
print(f"  W_hh 梯度矩阵:\n{rnn.weight_hh_l0.grad}")

In [ ]:
# 可视化RNN展开和BPTT
fig, ax = plt.subplots(figsize=(12, 6))

T = 4  # 时间步数
positions_x = np.arange(T) * 2.0
y_input = 2.0
y_rnn = 0.0
y_output = -2.0

# 画RNN单元
for t in range(T):
    # 输入箭头
    ax.annotate('', xy=(positions_x[t], y_rnn+0.3), xytext=(positions_x[t], y_input-0.3),
               arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    # 输入标签
    ax.text(positions_x[t], y_input+0.2, f'x{t}', ha='center', fontsize=12,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    # RNN单元
    ax.text(positions_x[t], y_rnn, 'RNN', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='circle', facecolor='lightyellow', alpha=0.8, edgecolor='orange', lw=2))
    # 输出箭头
    ax.annotate('', xy=(positions_x[t], y_output+0.3), xytext=(positions_x[t], y_rnn-0.3),
               arrowprops=dict(arrowstyle='->', color='red', lw=2))
    # 输出标签
    ax.text(positions_x[t], y_output-0.2, f'h{t+1}', ha='center', fontsize=12,
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    # 时间连接
    if t < T - 1:
        ax.annotate('', xy=(positions_x[t+1]-0.3, y_rnn+0.1), xytext=(positions_x[t]+0.3, y_rnn+0.1),
                   arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
        ax.text((positions_x[t]+positions_x[t+1])/2, y_rnn+0.35, 'h', ha='center', 
                fontsize=10, color='green', style='italic')

# BPTT标注
ax.text(3, -3.5, 'Forward Propagation -> (left to right)', fontsize=11, color='blue', style='italic')
ax.text(3, -4.0, 'Backward Propagation <- (right to left, BPTT)', fontsize=11, color='red', style='italic')
ax.annotate('', xy=(5.5, -3.4), xytext=(1.5, -3.4),
           arrowprops=dict(arrowstyle='->', color='blue', lw=1.5, linestyle='--'))
ax.annotate('', xy=(1.5, -3.9), xytext=(5.5, -3.9),
           arrowprops=dict(arrowstyle='->', color='red', lw=1.5, linestyle='--'))

# 共享参数标注
ax.text(3, y_rnn+1.2, 'All RNN cells share same parameters (W_ih, W_hh, b_ih, b_hh)',
        ha='center', fontsize=10, color='purple', style='italic',
        bbox=dict(boxstyle='round', facecolor='lavender', alpha=0.5))

ax.set_xlim(-1, T*2)
ax.set_ylim(-4.5, 3)
ax.axis('off')
ax.set_title('Unrolled RNN and BPTT', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 四、RNN的梯度问题

RNN在长序列上训练时面临严重的梯度问题。

### 为什么RNN更容易出现梯度消失？

在BPTT中，从时间步 $T$ 到时间步 $0$ 的梯度传播涉及**连乘**：

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

其中每一步：

$$\frac{\partial h_t}{\partial h_{t-1}} = W_{hh}^T \cdot \text{diag}(\tanh'(z_t))$$

由于：
1. $\tanh'(z)$ 的范围是 $(0, 1]$，通常 $< 1$
2. $W_{hh}$ 的特征值可能 $< 1$ 或 $> 1$

**连乘效应**导致：
- 如果特征值 $< 1$：长距离梯度趋近于0 → **梯度消失**
- 如果特征值 $> 1$：长距离梯度爆炸 → **梯度爆炸**

### 后果

- **梯度消失**：RNN无法学习**长距离依赖**（比如记住10步前的信息）
- **梯度爆炸**：训练不稳定，可能数值溢出

In [ ]:
# RNN梯度消失分析
print("=" * 70)
print("RNN梯度消失的数学分析")
print("=" * 70)
print()

# 模拟不同W_hh和序列长度下的梯度衰减
tanh_max_deriv = 0.7  # tanh导数平均约0.5~0.7
W_scales = [0.3, 0.5, 0.8, 1.0, 1.2]
seq_lengths = [5, 10, 20, 50, 100]

print("梯度衰减因子 (W_scale × tanh')^T")
print()
print(f"{'W_scale':<12}", end='')
for sl in seq_lengths:
    print(f"{'T='+str(sl):<12}", end='')
print()
print("-" * 72)

for ws in W_scales:
    decay_factor = ws * tanh_max_deriv
    print(f"{ws:<12.1f}", end='')
    for sl in seq_lengths:
        decay = decay_factor ** sl
        print(f"{decay:<12.2e}", end='')
    print()

print()
print("结论:")
print("  - W_scale=0.3: 20步后梯度衰减到 10^-18，几乎为0")
print("  - W_scale=1.0: 即使只有50步也衰减到 10^-15")
print("  - 这就是为什么普通RNN很难学习长距离依赖")

In [ ]:
# 实验: RNN能否学习长距离依赖？
print("\n=== 实验: RNN学习长距离依赖 ===")
print()

torch.manual_seed(42)

# 任务: 序列的第一个元素决定标签
# x = [signal, noise, noise, ..., noise]
# y = signal (第一个元素)
# 这是一个"记住第一个元素"的任务

def generate_long_memory_data(n_samples, seq_len, n_features=5):
    X = torch.randn(n_samples, seq_len, n_features)
    # 第一个元素决定标签
    signal = torch.randint(0, 2, (n_samples,)).float()
    X[:, 0, :] = signal.unsqueeze(1).expand(-1, n_features)
    y = signal
    return X, y

seq_len = 20
n_features = 5
X_train, y_train = generate_long_memory_data(1000, seq_len, n_features)
X_test, y_test = generate_long_memory_data(200, seq_len, n_features)

print(f"任务: 记住序列的第一个元素的值")
print(f"序列长度: {seq_len}")
print(f"特征维度: {n_features}")
print(f"训练样本: {X_train.shape[0]}")
print(f"标签分布: 0={int((y_train==0).sum())}, 1={int((y_train==1).sum())}")
print()

# 训练一个简单的RNN
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        output, h_n = self.rnn(x)
        return self.fc(h_n[-1]).squeeze(-1)

hidden_size = 16
model = SimpleRNN(n_features, hidden_size)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

losses = []
accuracies = []

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    losses.append(loss.item())
    
    # 计算准确率
    model.eval()
    with torch.no_grad():
        test_output = model(X_test)
        preds = (test_output > 0).float()
        acc = (preds == y_test).float().mean().item()
    accuracies.append(acc)
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc:.2%}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(losses, 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title(f'RNN Training Loss (seq_len={seq_len})')
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies, 'g-', linewidth=1.5)
axes[1].axhline(y=0.5, color='r', linestyle='--', label='随机猜测 (50%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('RNN Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n最终测试准确率: {accuracies[-1]:.2%}")
if accuracies[-1] < 0.8:
    print("普通RNN在序列长度20时表现不佳，难以学习长距离依赖！")
else:
    print("RNN成功学会了长距离依赖")

In [ ]:
# RNN擅长的任务：短距离序列预测
print("\n=== RNN序列预测实验 (正弦波) ===")
print()

torch.manual_seed(42)

# 生成正弦波数据
def generate_sine_data(seq_length=30, n_samples=500):
    X = []
    y = []
    for _ in range(n_samples):
        start = np.random.uniform(0, 4*math.pi)
        x_vals = np.linspace(start, start + seq_length*0.2, seq_length+1)
        sine_vals = np.sin(x_vals)
        X.append(sine_vals[:seq_length].reshape(-1, 1))
        y.append(sine_vals[seq_length])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)

seq_len = 30
X_train, y_train = generate_sine_data(seq_len, 500)
X_test, y_test = generate_sine_data(seq_len, 100)

# 训练RNN
class SineRNN(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        output, h_n = self.rnn(x)
        return self.fc(h_n[-1])

model = SineRNN(hidden_size=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

losses = []
for epoch in range(200):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = criterion(pred, y_train)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"训练完成，最终损失: {losses[-1]:.6f}")

# 可视化结果
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 损失曲线
axes[0].plot(losses, 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# 预测结果
model.eval()
with torch.no_grad():
    y_pred = model(X_test)

axes[1].scatter(y_test.numpy(), y_pred.numpy(), s=10, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('True Value')
axes[1].set_ylabel('Predicted Value')
axes[1].set_title(f'预测 vs 真实\n(R² = {1 - torch.sum((y_test-y_pred)**2)/torch.sum((y_test-y_test.mean())**2):.4f})')
axes[1].grid(True, alpha=0.3)

# 展示一个样本
sample_idx = 0
sample_input = X_test[sample_idx].numpy().flatten()
sample_pred = y_pred[sample_idx].item()
sample_true = y_test[sample_idx].item()

all_x = list(range(seq_len)) + [seq_len]
all_y = list(sample_input) + [sample_true]
pred_y = list(sample_input) + [sample_pred]

axes[2].plot(range(seq_len), sample_input, 'b-o', markersize=4, linewidth=1.5, label='输入序列')
axes[2].plot(seq_len, sample_true, 'g*', markersize=15, label=f'真实值={sample_true:.3f}')
axes[2].plot(seq_len, sample_pred, 'r^', markersize=15, label=f'预测值={sample_pred:.3f}')
axes[2].set_xlabel('Time Step')
axes[2].set_ylabel('sin value')
axes[2].set_title('Single Sample Prediction')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 五、本章总结

### 核心要点

1. **RNN的核心思想**：用隐藏状态记录历史信息，具有记忆能力
   - $h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$
   - 同一组参数在所有时间步共享

2. **BPTT**：将RNN按时间展开后应用反向传播
   - $W_{hh}$ 的梯度是所有时间步贡献的总和

3. **RNN的致命缺陷：梯度消失**
   - $\tanh$ 导数 $< 1$ 连乘导致梯度趋近于0
   - 无法学习长距离依赖（>10步就很困难）

4. **RNN适合的任务**：短距离序列预测（如正弦波预测）

### 解决梯度消失的方向

为了解决RNN的梯度消失问题，研究者提出了：
- **LSTM（长短期记忆）**：引入门控机制，保护梯度流
- **GRU（门控循环单元）**：LSTM的简化版，效率更高

### 下一篇预告

下一篇将学习**LSTM和GRU**——它们如何通过门控机制解决RNN的梯度消失问题。